<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Baseline_AttentionUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import matplotlib.pyplot as plt

In [3]:
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [4]:
import numpy as np
import tensorflow as tf

# 1. Define the ESA class map
class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# 2. Identify which classes actually exist in your data
unique_labels = sorted(np.unique(Y_train_mask))
label_map = {old: new for new, old in enumerate(unique_labels)}

print("--- LABEL MAPPING TABLE ---")
for old_id, new_id in label_map.items():
    name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"Original ESA ID: {old_id:2} ({name:15}) -> New Neural Net ID: {new_id}")

# 3. Create the Look-Up Table (LUT) - THE FIX
# This maps the high ESA numbers to 0, 1, 2, 3, 4, 5 instantly across all pixels
lut = np.zeros(91, dtype=np.int32)
for old_id, new_id in label_map.items():
    lut[old_id] = new_id

# 4. Apply mapping to the whole 3D array (Spatial Mapping)
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

--- LABEL MAPPING TABLE ---
Original ESA ID: 10 (Tree cover     ) -> New Neural Net ID: 0
Original ESA ID: 20 (Shrubland      ) -> New Neural Net ID: 1
Original ESA ID: 30 (Grassland      ) -> New Neural Net ID: 2
Original ESA ID: 40 (Cropland       ) -> New Neural Net ID: 3
Original ESA ID: 50 (Built-up       ) -> New Neural Net ID: 4
Original ESA ID: 60 (Bare / Sparse vegetation) -> New Neural Net ID: 5
Original ESA ID: 80 (Permanent water bodies) -> New Neural Net ID: 6
Original ESA ID: 90 (Herbaceous wetland) -> New Neural Net ID: 7


In [5]:
# 5. One-Hot Encode for U-Net
num_classes = len(unique_labels)
Y_train_cat = tf.keras.utils.to_categorical(Y_train_ready, num_classes=num_classes)
Y_test_cat = tf.keras.utils.to_categorical(Y_test_ready, num_classes=num_classes)

print(f"X_train shape: {X_train.shape}") # no.of bands(7)
print(f"Y_train shape: {Y_train_cat.shape}") # no.of classes(8)

X_train shape: (639, 256, 256, 7)
Y_train shape: (639, 256, 256, 8)


In [6]:
# 1. Check the Data Type
print(f"X_train dtype: {X_train.dtype}")
print(f"Y_train_cat dtype: {Y_train_cat.dtype}")
print(f"X_test dtype: {X_test.dtype}")
print(f"Y_test_cat dtype: {Y_test_cat.dtype}")

# 2. Check Memory Usage (in Gigabytes)
# .nbytes gives the total bytes consumed by the elements of the array
x_mem = X_train.nbytes / (1024**3)
y_mem = Y_train_cat.nbytes / (1024**3)
b_mem = X_test.nbytes / (1024**3)
a_mem = Y_test_cat.nbytes / (1024**3)

print(f"\nX_train is using: {x_mem:.2f} GB")
print(f"Y_train_cat is using: {y_mem:.2f} GB")
print(f"X_test is using: {b_mem:.2f} GB")
print(f"Y_test_cat is using: {a_mem:.2f} GB")
print(f"Total RAM for training set: {x_mem + y_mem + b_mem + a_mem:.2f} GB")

X_train dtype: float64
Y_train_cat dtype: float64
X_test dtype: float64
Y_test_cat dtype: float64

X_train is using: 2.18 GB
Y_train_cat is using: 2.50 GB
X_test is using: 0.43 GB
Y_test_cat is using: 0.49 GB
Total RAM for training set: 5.60 GB


In [7]:
X_train = X_train.astype('float32')
Y_train_cat = Y_train_cat.astype('float32')
X_test = X_test.astype('float32')
Y_test_cat = Y_test_cat.astype('float32')

print(f"New Total RAM: {(X_train.nbytes + Y_train_cat.nbytes) / (1024**3):.2f} GB")

New Total RAM: 2.34 GB


In [8]:
from tensorflow.keras import layers, models, backend as K

def attention_block(x, gating, inter_shape):
    """
    x: skip connection from encoder (e.g., 128x128)
    gating: gating signal from decoder (e.g., 128x128)
    """
    # 1. Project skip connection (x) - Keep same size
    theta_x = layers.Conv2D(inter_shape, (1, 1), padding='same')(x)

    # 2. Project gating signal (gating) - Keep same size
    phi_g = layers.Conv2D(inter_shape, (1, 1), padding='same')(gating)

    # 3. Add them together (They are now both the same spatial size)
    concat_xg = layers.add([phi_g, theta_x])
    act_xg = layers.Activation('relu')(concat_xg)

    # 4. Calculate attention coefficients (0 to 1)
    psi = layers.Conv2D(1, (1, 1), padding='same')(act_xg)
    sigmoid_xg = layers.Activation('sigmoid')(psi)

    # 5. Multiply the original skip connection by the attention map
    y = layers.multiply([sigmoid_xg, x])

    # 6. Final cleanup with 1x1 convolution
    result = layers.Conv2D(K.int_shape(x)[3], (1, 1), padding='same')(y)
    result_bn = layers.BatchNormalization()(result)
    return result_bn

def build_attention_unet(input_shape=(256, 256, 7), num_classes=8):
    inputs = layers.Input(input_shape)

    # --- ENCODER ---
    # Level 1: 256x256
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    # Level 2: 128x128
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Bridge (Bottom): 64x64
    b1 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    b1 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(b1)

    # --- DECODER with Attention ---
    # Up 1: 64x64 -> 128x128
    # gating signal g1 will be 128x128
    g1 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(b1)
    # x=c2 is also 128x128. Perfect match!
    a1 = attention_block(x=c2, gating=g1, inter_shape=128)
    u1 = layers.concatenate([g1, a1])
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u1)

    # Up 2: 128x128 -> 256x256
    # gating signal g2 will be 256x256
    g2 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c3)
    # x=c1 is also 256x256. Perfect match!
    a2 = attention_block(x=c1, gating=g2, inter_shape=64)
    u2 = layers.concatenate([g2, a2])
    c4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u2)

    # Output Layer
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c4)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model


In [9]:
import tensorflow as tf

def categorical_focal_loss(gamma=2.0, alpha=0.25):
    """
    Implementation of Focal Loss for multi-class segmentation.
    Focuses on 'hard' pixels (rare classes) and ignores 'easy' pixels.
    """
    def focal_loss(y_true, y_pred):
        # Clip predictions to prevent NaN/Log(0)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        # Calculate Cross Entropy
        cross_entropy = -y_true * tf.math.log(y_pred)

        # Calculate Weighting Factor (1 - p)^gamma
        # If the model is confident (p is high), weight is low.
        # If the model is unsure (p is low), weight is high.
        loss = tf.pow(1.0 - y_pred, gamma) * cross_entropy

        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

    return focal_loss

In [10]:
# Re-build and Compile
model = build_attention_unet(input_shape=(256, 256, 7), num_classes=8)

# Compile with Focal Loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=categorical_focal_loss(gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.OneHotMeanIoU(num_classes=8, name='iou')]
)

print("Model Built successfully")

Model Built successfully


In [11]:
import numpy as np
from sklearn.metrics import f1_score

class SegmentationLogger(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.X_val, self.y_val_cat = val_data
        # Convert one-hot back to integers for f1_score comparison
        self.y_val_true = np.argmax(self.y_val_cat, axis=-1).flatten()

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Loss':<8} | {'Val Loss':<10} | {'Val Acc':<8} | {'Val F1':<8}")
        print("-" * 55)

    def on_epoch_end(self, epoch, logs=None):
        # Predict on validation set
        val_preds_probs = self.model.predict(self.X_val, verbose=0)
        val_preds_ints = np.argmax(val_preds_probs, axis=-1).flatten()

        # Calculate Macro F1 (Average across all 8 classes)
        val_f1 = f1_score(self.y_val_true, val_preds_ints, average='macro', zero_division=0)

        loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('accuracy', 0)

        print(f"{epoch+1:<6} | {loss:<8.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {val_f1:<8.4f}")


In [12]:
logger = SegmentationLogger(val_data=(X_test, Y_test_cat))

# Setup basic training parameters
history = model.fit(
    X_train, Y_train_cat,
    batch_size=16,
    epochs=10,
    validation_data=(X_test, Y_test_cat),
    callbacks=[logger],
    shuffle=True,
    verbose=0
)


Epoch  | Loss     | Val Loss   | Val Acc  | Val F1  
-------------------------------------------------------
1      | 0.8910   | 1.4265     | 0.5513   | 0.0636  
2      | 0.4702   | 1.3869     | 0.6488   | 0.0464  
3      | 0.3906   | 1.4732     | 0.6869   | 0.0450  
4      | 0.3612   | 1.3576     | 0.7014   | 0.0450  
5      | 0.3606   | 1.2068     | 0.7018   | 0.0453  
6      | 0.3316   | 1.1441     | 0.7221   | 0.0450  
7      | 0.3333   | 0.9586     | 0.7159   | 0.0492  
8      | 0.3240   | 0.8310     | 0.7251   | 0.0685  
9      | 0.3170   | 0.7809     | 0.7247   | 0.2223  
10     | 0.3137   | 0.6139     | 0.7285   | 0.3884  


In [13]:
from sklearn.metrics import classification_report

print("Generating final pixel-wise predictions...")
test_probs = model.predict(X_test, batch_size=16)
test_preds = np.argmax(test_probs, axis=-1).flatten()
test_true = np.argmax(Y_test_cat, axis=-1).flatten()

# Get class names in order
target_names = [class_map[old_id][0] for old_id, new_id in sorted(label_map.items(), key=lambda x: x[1])]

print("\n--- PIXEL-WISE CLASSIFICATION REPORT ---")
print(classification_report(test_true, test_preds, target_names=target_names, zero_division=0))

Generating final pixel-wise predictions...
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 336ms/step

--- PIXEL-WISE CLASSIFICATION REPORT ---
                          precision    recall  f1-score   support

              Tree cover       0.83      0.40      0.54    829607
               Shrubland       0.64      0.14      0.23   1016613
               Grassland       0.33      0.90      0.49   1772697
                Cropland       0.90      0.00      0.00   1774223
                Built-up       0.81      0.89      0.85   1090768
Bare / Sparse vegetation       0.04      0.01      0.01     48484
  Permanent water bodies       1.00      0.96      0.98   1653581
      Herbaceous wetland       0.00      0.00      0.00      6027

                accuracy                           0.57   8192000
               macro avg       0.57      0.41      0.39   8192000
            weighted avg       0.74      0.57      0.50   8192000

